In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("dataset/cs-training.csv")

if df.columns[0].startswith("Unnamed") or df.columns[0] == "":
    df = df.rename(columns={df.columns[0]: "applicant_id"})

if "applicant_id" not in df.columns:
    raise ValueError("Dataset must contain an applicant_id column.")

if df["applicant_id"].isna().any() or df["applicant_id"].duplicated().any():
    raise ValueError("Applicant IDs must be present and unique.")

X = df.drop(columns=["applicant_id", "SeriousDlqin2yrs"])
y = df["SeriousDlqin2yrs"]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    random_state=42,
    stratify=y_train_val
)

In [2]:
for name, features, target in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Test", X_test, y_test)
]:
    print(
        f"{name}: {len(features):,} rows | "
        f"Positive outcomes: {target.mean():.2%}"
    )

print("\nMissing values in training data:")
print(X_train.isna().sum())

Training: 90,000 rows | Positive outcomes: 6.68%
Validation: 30,000 rows | Positive outcomes: 6.68%
Test: 30,000 rows | Positive outcomes: 6.68%

Missing values in training data:
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           17796
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       2309
dtype: int64


In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['RevolvingUtilizationOfUnsecuredLines','age', 'NumberOfTime30-59DaysPastDueNotWorse',...,'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse','NumberOfDependents']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time

In [4]:
val_probabilities = logistic_model.predict_proba(X_val)[:, 1]

print("First five probabilities:")
print(val_probabilities[:5])

First five probabilities:
[0.09320787 0.02259466 0.0427884  0.07326841 0.11999128]


In [5]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

val_predictions = (val_probabilities >= 0.50).astype(int)

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val, val_probabilities), 4))

print("Validation average precision:",
      round(average_precision_score(y_val, val_probabilities), 4))

print("\nClassification report:")
print(classification_report(
    y_val,
    val_predictions,
    digits=3,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_val, val_predictions))

Validation ROC-AUC: 0.6918
Validation average precision: 0.2267

Classification report:
              precision    recall  f1-score   support

           0      0.935     0.998     0.966     27995
           1      0.567     0.038     0.071      2005

    accuracy                          0.934     30000
   macro avg      0.751     0.518     0.518     30000
weighted avg      0.911     0.934     0.906     30000

Confusion matrix:
[[27937    58]
 [ 1929    76]]


In [6]:
from sklearn.ensemble import RandomForestClassifier

forest_model = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ))
])

forest_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['RevolvingUtilizationOfUnsecuredLines','age', 'NumberOfTime30-59DaysPastDueNotWorse',...,'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse','NumberOfDependents']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.

In [7]:
forest_val_probabilities = forest_model.predict_proba(X_val)[:, 1]

forest_val_predictions = (
    forest_val_probabilities >= 0.50
).astype(int)

print("Validation ROC-AUC:",
      round(roc_auc_score(y_val, forest_val_probabilities), 4))

print("Validation average precision:",
      round(average_precision_score(
          y_val, forest_val_probabilities
      ), 4))

print("\nClassification report:")
print(classification_report(
    y_val,
    forest_val_predictions,
    digits=3,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_val, forest_val_predictions))

Validation ROC-AUC: 0.8657
Validation average precision: 0.387

Classification report:
              precision    recall  f1-score   support

           0      0.942     0.993     0.967     27995
           1      0.607     0.151     0.242      2005

    accuracy                          0.937     30000
   macro avg      0.775     0.572     0.605     30000
weighted avg      0.920     0.937     0.919     30000

Confusion matrix:
[[27799   196]
 [ 1702   303]]


In [9]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = []

for threshold in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    predictions = (
        forest_val_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions
    ).ravel()

    results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "detected": tp,
        "missed": fn,
        "false_alarms": fp
    })

threshold_results = pd.DataFrame(results)
print(threshold_results.round(3).to_string(index=False))

 threshold  precision  recall    f1  detected  missed  false_alarms
      0.05      0.180   0.828 0.296      1661     344          7551
      0.10      0.269   0.673 0.384      1349     656          3669
      0.15      0.338   0.570 0.424      1142     863          2241
      0.20      0.385   0.498 0.435       999    1006          1594
      0.25      0.430   0.444 0.437       890    1115          1182
      0.30      0.461   0.384 0.419       770    1235           900
      0.40      0.536   0.261 0.351       523    1482           453
      0.50      0.607   0.151 0.242       303    1702           196


In [10]:
selected_model = forest_model
selected_threshold = 0.25

test_probabilities = selected_model.predict_proba(X_test)[:, 1]

test_predictions = (
    test_probabilities >= selected_threshold
).astype(int)

print("Selected threshold:", selected_threshold)

print("Test ROC-AUC:",
      round(roc_auc_score(y_test, test_probabilities), 4))

print("Test average precision:",
      round(average_precision_score(
          y_test, test_probabilities
      ), 4))

print("\nClassification report:")
print(classification_report(
    y_test,
    test_predictions,
    digits=3,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, test_predictions))

Selected threshold: 0.25
Test ROC-AUC: 0.8676
Test average precision: 0.4025

Classification report:
              precision    recall  f1-score   support

           0      0.961     0.957     0.959     27995
           1      0.429     0.453     0.441      2005

    accuracy                          0.923     30000
   macro avg      0.695     0.705     0.700     30000
weighted avg      0.925     0.923     0.924     30000

Confusion matrix:
[[26788  1207]
 [ 1097   908]]


In [11]:
from pathlib import Path
import joblib
import sklearn

Path("models").mkdir(exist_ok=True)

model_bundle = {
    "pipeline": selected_model,
    "threshold": selected_threshold,
    "feature_columns": X_train.columns.tolist(),
    "target": "SeriousDlqin2yrs",
    "threshold_selection": "Highest validation F1 among tested thresholds",
    "sklearn_version": sklearn.__version__,
    "test_metrics": {
        "roc_auc": float(roc_auc_score(
            y_test, test_probabilities
        )),
        "average_precision": float(average_precision_score(
            y_test, test_probabilities
        )),
        "precision": float(precision_score(
            y_test, test_predictions, zero_division=0
        )),
        "recall": float(recall_score(
            y_test, test_predictions, zero_division=0
        )),
        "f1": float(f1_score(
            y_test, test_predictions, zero_division=0
        ))
    }
}

joblib.dump(model_bundle, "models/credit_risk_model.joblib")

print("Saved models/credit_risk_model.joblib")

Saved models/credit_risk_model.joblib


In [1]:
%pip install shap

  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/755.4 kB ? eta -:--:--
   ---------------------------------------- 755.4/755.4 kB 6.1 MB/s  0:00:00
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)

   ------------- -------------------------- 1/3 [cloudpickle]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   -------------------------- ------------- 2/3 [shap]
   --


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import joblib
import numpy as np
import pandas as pd
import shap

bundle = joblib.load("models/credit_risk_model.joblib")
pipeline = bundle["pipeline"]

df = pd.read_csv("dataset/cs-training.csv")

if df.columns[0].startswith("Unnamed") or df.columns[0] == "":
    df = df.rename(columns={df.columns[0]: "applicant_id"})

applicant_input = df.loc[
    df["applicant_id"] == 1,
    bundle["feature_columns"]
]

imputer = pipeline.named_steps["imputer"]
forest = pipeline.named_steps["classifier"]

prepared_input = imputer.transform(applicant_input)
feature_names = imputer.get_feature_names_out(
    bundle["feature_columns"]
)

c:\Users\Lenovo\MLProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
explainer = shap.TreeExplainer(
    forest,
    feature_perturbation="tree_path_dependent",
    model_output="raw"
)

explanation = explainer(prepared_input)

positive_class_index = int(
    np.flatnonzero(forest.classes_ == 1)[0]
)

contributions = explanation.values[0, :, positive_class_index]
baseline = float(
    explanation.base_values[0, positive_class_index]
)

probability = float(
    pipeline.predict_proba(applicant_input)[0, positive_class_index]
)

reconstructed_probability = baseline + contributions.sum()

print(f"Baseline: {baseline:.6f}")
print(f"Sum of contributions: {contributions.sum():.6f}")
print(f"Reconstructed probability: {reconstructed_probability:.6f}")
print(f"Model probability: {probability:.6f}")

np.testing.assert_allclose(
    reconstructed_probability,
    probability,
    atol=1e-6,
    rtol=1e-6
)

print("SHAP reconstruction check passed.")

Baseline: 0.066779
Sum of contributions: 0.261310
Reconstructed probability: 0.328089
Model probability: 0.328089
SHAP reconstruction check passed.


In [4]:
contribution_table = pd.DataFrame({
    "feature": feature_names,
    "model_input_value": prepared_input[0],
    "contribution_percentage_points": contributions * 100
})

contribution_table = contribution_table.iloc[
    np.argsort(-np.abs(contributions))
]

print(contribution_table.round(4).to_string(index=False))

                             feature  model_input_value  contribution_percentage_points
NumberOfTime30-59DaysPastDueNotWorse             2.0000                         10.9805
RevolvingUtilizationOfUnsecuredLines             0.7661                          7.8548
        NumberRealEstateLoansOrLines             6.0000                          5.7221
                           DebtRatio             0.8030                          3.8033
             NumberOfTimes90DaysLate             0.0000                         -2.1349
NumberOfTime60-89DaysPastDueNotWorse             0.0000                         -1.1229
                                 age            45.0000                          0.8910
                       MonthlyIncome          9120.0000                         -0.8219
                  NumberOfDependents             2.0000                          0.5315
     NumberOfOpenCreditLinesAndLoans            13.0000                          0.2715
      missingindicator_MonthlyIn

In [5]:
missing_income_id = df.loc[
    df["MonthlyIncome"].isna(), "applicant_id"
].iloc[0]

check_ids = list(dict.fromkeys([
    1, 2, 3, int(missing_income_id)
]))

check_input = df.loc[
    df["applicant_id"].isin(check_ids),
    ["applicant_id"] + bundle["feature_columns"]
]

prepared_checks = imputer.transform(
    check_input[bundle["feature_columns"]]
)

check_explanations = explainer(prepared_checks)

check_contributions = check_explanations.values[
    :, :, positive_class_index
]

check_baselines = check_explanations.base_values[
    :, positive_class_index
]

reconstructed = (
    check_baselines + check_contributions.sum(axis=1)
)

predicted = pipeline.predict_proba(
    check_input[bundle["feature_columns"]]
)[:, positive_class_index]

np.testing.assert_allclose(
    reconstructed,
    predicted,
    atol=1e-6,
    rtol=1e-6
)

print(pd.DataFrame({
    "applicant_id": check_input["applicant_id"].to_numpy(),
    "model_probability": predicted,
    "shap_reconstructed": reconstructed,
    "income_missing": check_input["MonthlyIncome"].isna().to_numpy()
}).to_string(index=False))

print("All reconstruction checks passed.")

 applicant_id  model_probability  shap_reconstructed  income_missing
            1           0.328089            0.328089           False
            2           0.091929            0.091929           False
            3           0.345241            0.345241           False
            7           0.033512            0.033512            True
All reconstruction checks passed.
